### **Purpose: Setting Up Our Tools**

This cell is like preparing our workbench before starting a project. We use the `!pip install -q` command to download and install necessary Python libraries. Think of these libraries as specialized toolkits that help us do specific tasks more easily.

*   **`langchain`**: A framework that helps us build applications powered by large language models (LLMs).
*   **`langchain-openai`**: A specific connector to use OpenAI's powerful AI models through the LangChain framework.
*   **`langgraph`**: A library for building robust and stateful multi-actor applications with LLMs.
*   **`requests`**: Used to send requests to websites and get data back (like job postings).
*   **`pypdf`**: Helps us read and extract text from PDF files, which is useful for processing resumes.
*   **`pandas`**: A fundamental library for data manipulation and analysis, making it easy to work with data in tables.

In [1]:
!pip install -q langchain langchain-openai langgraph requests pypdf pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 513.3/513.3 kB 28.6 MB/s eta 0:00:00


### **Purpose: Importing Libraries and Initializing AI**

This cell is about getting our tools ready and setting up the core AI model we'll be using.

1.  **Importing Libraries**: We bring in various Python libraries (`os`, `json`, `requests`, `pandas`, `PdfReader`, `files`, `Markdown`, `display`, `ChatOpenAI`, `tool`, `create_agent`) that provide functions for interacting with the operating system, handling data, reading PDFs, displaying rich output, and working with AI agents.
2.  **OpenAI API Key Setup**: We store our secret OpenAI API key. This key is like a password that allows our program to talk to OpenAI's powerful language models. It's kept secure so that only our program can use it.
3.  **Initializing the Language Model (LLM)**: We create an instance of `ChatOpenAI`. This is our main AI brain. We've chosen a specific model (`gpt-5.4-mini`) and set its `temperature` to 0, which means the AI will be very consistent and less creative in its responses, ideal for structured tasks like job ranking.
4.  **`extract_text_from_pdf` Function**: This helper function is designed to open any PDF file (like a resume), read through all its pages, and extract all the visible text. This way, we can get the content of a resume in a format the AI can understand.

In [2]:
import os
import json
import requests, random
import pandas as pd
from pypdf import PdfReader
from google.colab import files
from IPython.display import Markdown, display

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# Paste your OpenAI API key here
OPENAI_API_KEY = "Paste your key here"
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# LangChain chat model
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# Function to read resume text from PDF
def extract_text_from_pdf(file_name):
    reader = PdfReader(file_name)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text.strip()

### **Purpose: Defining Job Search Platforms and Fetching Jobs**

This cell is crucial for collecting job postings from various online platforms.

1.  **Company Lists**: We define lists of popular companies that use specific applicant tracking systems (ATS) like Greenhouse, Lever, and Ashby. This helps us know where to look for jobs.
2.  **Job Fetching Tools (`@tool`)**: We create special functions, marked with `@tool`, which act like smart agents capable of interacting with these job platforms.
    *   **`fetch_greenhouse_jobs`**: This tool connects to the Greenhouse API to find job postings. It searches for jobs that match a given role and location within companies using Greenhouse.
    *   **`fetch_lever_jobs`**: Similar to Greenhouse, this tool fetches jobs from companies using the Lever ATS.
    *   **`fetch_ashby_jobs`**: This tool retrieves job postings from companies utilizing the Ashby platform.
    Each tool takes a `job_role` and `location` as input, sends a request to the respective platform, and extracts relevant job details like title, company, location, and URL.
3.  **`fetch_all_jobs` Function**: This function combines the power of all three individual job-fetching tools. It calls each tool, gathers all the jobs found, removes any duplicate entries (to avoid showing the same job multiple times), and limits the results to the top 30 jobs. This way, we get a comprehensive list of job openings relevant to our search criteria.

In [3]:
GREENHOUSE_COMPANIES = [
    "airbnb", "dropbox", "reddit", "notion", "figma", "stripe",
    "coinbase", "duolingo", "hubspot", "asana", "zendesk",
    "twilio", "datadog", "snowflake", "pinterest", "squarespace"
]

LEVER_COMPANIES = [
    "netflix", "uber", "lyft", "atlassian", "canva", "shopify",
    "cloudflare", "plaid", "brex", "airtable", "retool", "loom"
]


@tool
def fetch_greenhouse_jobs(job_role: str, location: str):
    """Fetch jobs from Greenhouse based on job role and location."""
    jobs = []
    for company in GREENHOUSE_COMPANIES:
        try:
            res = requests.get(
                f"https://boards-api.greenhouse.io/v1/boards/{company}/jobs",
                timeout=5
            )
            if res.status_code == 200:
                for job in res.json().get("jobs", []):
                    title = job.get("title", "")
                    loc = job.get("location", {}).get("name", "Not specified")

                    # Greenhouse has good location data so we keep the filter
                    if job_role.lower() in title.lower() and (
                        location.lower() in loc.lower() or "remote" in loc.lower()
                    ):
                        jobs.append({
                            "title": title,
                            "company": company.capitalize(),
                            "location": loc,
                            "url": job.get("absolute_url", ""),
                            "source": "Greenhouse"
                        })
            else:
                print(f"Greenhouse: Failed for {company}. Status: {res.status_code}")
        except Exception as e:
            print(f"Greenhouse: Error for {company}: {e}")
            continue
    print(f"Greenhouse fetched {len(jobs)} jobs.")
    return jobs


@tool
def fetch_lever_jobs(job_role: str, location: str):
    """Fetch jobs from Lever based on job role."""
    jobs = []
    for company in LEVER_COMPANIES:
        try:
            res = requests.get(
                f"https://api.lever.co/v0/postings/{company}?mode=json",
                timeout=5
            )
            if res.status_code == 200:
                for job in res.json():
                    title = job.get("text", "")
                    loc = job.get("categories", {}).get("location", "Not specified")

                    # Lever location data is inconsistent so we only filter by role
                    if job_role.lower() in title.lower():
                        jobs.append({
                            "title": title,
                            "company": company.capitalize(),
                            "location": loc,
                            "url": job.get("hostedUrl", ""),
                            "source": "Lever"
                        })
            else:
                print(f"Lever: Failed for {company}. Status: {res.status_code}")
        except Exception as e:
            print(f"Lever: Error for {company}: {e}")
            continue
    print(f"Lever fetched {len(jobs)} jobs.")
    return jobs



def fetch_all_jobs(job_role, location):
    print(f"Fetching Greenhouse jobs for role='{job_role}', location='{location}'")
    greenhouse_jobs = fetch_greenhouse_jobs.invoke({"job_role": job_role, "location": location})

    print(f"Fetching Lever jobs for role='{job_role}', location='{location}'")
    lever_jobs = fetch_lever_jobs.invoke({"job_role": job_role, "location": location})

    print(f"\nGreenhouse: {len(greenhouse_jobs)} found")
    print(f"Lever: {len(lever_jobs)} found")

    combined = greenhouse_jobs + lever_jobs

    # Remove duplicates by URL
    seen = set()
    final_jobs = []
    for job in combined:
        if job["url"] and job["url"] not in seen:
            seen.add(job["url"])
            final_jobs.append(job)

    # Shuffle so LLM doesn't just pick the first ones it sees
    random.shuffle(final_jobs)

    print(f"Total unique jobs found: {len(final_jobs)}")
    return final_jobs

print("Job tools ready!")

Job tools ready!


### **Purpose: Uploading Resume and Initial Job Search**

This cell handles the user's input and kicks off the job search process.

1.  **`files.upload()`**: This command allows you to upload your resume directly into the Colab environment. It prompts a file selection dialog.
2.  **Extracting Resume Text**: Once uploaded, we use the `extract_text_from_pdf` function (defined earlier) to read your resume PDF and convert all its content into plain text. This text will be used by our AI to understand your skills and experience.
3.  **User Input for Preferences**: We then ask you to provide your `preferred_job_role` (e.g., 'Data Scientist') and `preferred_location` (e.g., 'New York'). This helps narrow down the job search.
4.  **Fetching All Jobs**: The `fetch_all_jobs` function (defined in the previous cell) is called with your specified role and location. It goes out to the various job boards and collects relevant postings.
5.  **Confirmation and Sample**: Finally, the cell prints a confirmation that your resume was processed and how many jobs were found. If jobs are available, it displays a sample job entry so you can see the format of the collected data.

In [4]:
uploaded = files.upload()

resume_file_name = list(uploaded.keys())[0]
resume_text = extract_text_from_pdf(resume_file_name)

preferred_role = input("Enter preferred job role: ")
preferred_location = input("Enter preferred location: ")

jobs = fetch_all_jobs(preferred_role, preferred_location)

print("Resume uploaded successfully.")
print("Jobs fetched:", len(jobs))

if jobs:
    print(" ")
else:
    print("\nNo jobs found. Try a broader role or location.")

Saving Riya_resume.pdf to Riya_resume.pdf
Enter preferred job role: data engineer
Enter preferred location: united states
Fetching Greenhouse jobs for role='data engineer', location='united states'
Greenhouse: Failed for notion. Status: 404
Greenhouse: Failed for zendesk. Status: 404
Greenhouse: Failed for snowflake. Status: 404
Greenhouse fetched 11 jobs.
Fetching Lever jobs for role='data engineer', location='united states'
Lever: Error for netflix: HTTPSConnectionPool(host='api.lever.co', port=443): Read timed out. (read timeout=5)
Lever: Failed for uber. Status: 404
Lever: Failed for lyft. Status: 404
Lever: Error for canva: HTTPSConnectionPool(host='api.lever.co', port=443): Read timed out. (read timeout=5)
Lever: Failed for shopify. Status: 404
Lever: Failed for cloudflare. Status: 404
Lever: Error for plaid: HTTPSConnectionPool(host='api.lever.co', port=443): Read timed out. (read timeout=5)
Lever: Failed for brex. Status: 404
Lever: Failed for airtable. Status: 404
Lever: Faile

### **Purpose: Extracting Keywords and Ranking Jobs with AI**

This cell is where the AI does its main work: understanding your resume and ranking job opportunities.

1.  **`get_resume_keywords` Function**:
    *   This function takes your resume text and sends it to our AI (`llm`).
    *   It asks the AI to act like an expert and identify the **10 most important job-related keywords** from your resume. These keywords represent your core skills and experience (e.g., 'Python', 'SQL', 'Machine Learning').
    *   The AI is instructed to return these keywords in a structured JSON list, making it easy for our program to use.
2.  **`rank_jobs` Function**:
    *   This is the core ranking engine. It takes your resume text, the keywords extracted by the AI, and the list of fetched jobs.
    *   It sends all this information to the AI (`llm`) with a detailed instruction: "Compare the jobs with the resume and keywords, rank the best 10 jobs, keep the exact URL, give a short reason for the ranking, and mention 1-3 missing skills if relevant."
    *   The AI acts as a sophisticated job matcher, evaluating how well each job description aligns with your profile and suggesting areas for improvement. It returns the ranked jobs in a structured JSON format, including a `match_score`, a `reason` for the ranking, and `missing_skills`.
3.  **Execution**: The cell then calls these two functions: first to get the `keywords` from your resume, and then to get the `ranked_jobs` using those keywords and the previously fetched jobs. Finally, it prints the extracted keywords for your review.

In [5]:
def get_resume_keywords(resume_text):
    prompt = f"""
Read this resume and extract the 20 most important job-related keywords.

Return ONLY a valid JSON list like:
["Python", "SQL", "Tableau"]

Resume:
{resume_text[:5000]}
"""
    response = llm.invoke(prompt)
    text = response.content.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(text)


def rank_jobs(resume_text, keywords, jobs):
    prompt = f"""
A student has this resume:
{resume_text[:5000]}

Important resume keywords:
{keywords}

These are the jobs:
{json.dumps(jobs)}

Task:
1. Compare the jobs with the resume and keywords
2. Rank the best 20 jobs. Be sure to have jobs from all the sources. If you feel that, more than 20 jobs are strong for the resume, then rank all of those.
3. Keep the exact original URL
4. Give a short reason
5. Mention 1-3 missing skills if relevant

Return ONLY valid JSON in this format:
[
  {{
    "rank": 1,
    "title": "...",
    "company": "...",
    "location": "...",
    "source": "...",
    "match_score": 90,
    "reason": "...",
    "missing_skills": ["...", "..."],
    "url": "..."
  }}
]
"""
    response = llm.invoke(prompt)
    text = response.content.strip().replace("```json", "").replace("```", "").strip()
    return json.loads(text)


keywords = get_resume_keywords(resume_text)
ranked_jobs = rank_jobs(resume_text, keywords, jobs)

print("Resume Keywords:")
print(keywords)

Resume Keywords:
['Python', 'SQL', 'R', 'Scala', 'PySpark', 'Apache Spark', 'Kafka', 'Hadoop', 'Hive', 'dbt', 'Alteryx', 'GCP', 'Snowflake', 'Databricks', 'Azure', 'AWS', 'Power BI', 'Tableau', 'Git', 'ETL']


### **Purpose: Displaying Ranked Jobs in an Interactive Table**

This final cell takes the AI's ranked job list and presents it in a user-friendly, interactive format.

1.  **Importing `display` and `HTML`**: We bring in tools from `IPython.display` that allow us to show rich content (like HTML) directly in the notebook.
2.  **Creating a DataFrame**: The `ranked_jobs` list (which is a list of dictionaries, perfect for tabular data) is converted into a `pandas.DataFrame`. Think of a DataFrame as a powerful spreadsheet that makes it easy to organize and manipulate data.
3.  **Making URLs Clickable**: The `make_clickable` function transforms the job URLs into actual clickable links. This is a small but important detail for usability, allowing you to directly open job postings from the table.
4.  **Applying Clickable Links**: This function is then applied to the 'url' column of our DataFrame, turning plain text URLs into interactive links.
5.  **Converting to HTML**: The DataFrame is converted into an HTML table string. `escape=False` is important here because we want our clickable links (which are HTML) to be rendered as-is, not as plain text.
6.  **Displaying the Table**: Finally, `display(HTML(html_table))` renders the HTML table directly in your Colab output. This gives you a clear, sortable, and clickable list of job recommendations based on your resume and preferences.

In [6]:
from IPython.display import display, HTML

df = pd.DataFrame(ranked_jobs)

def make_clickable(url):
    return f'<a href="{url}" target="_blank">Open Job</a>'

if "url" in df.columns:
    df["url"] = df["url"].apply(make_clickable)

# Convert DataFrame to HTML and display
html_table = df.to_html(escape=False, index=False)

display(HTML(html_table))

rank,title,company,location,source,match_score,reason,missing_skills,url
1,"Senior Staff Data Engineer, Marketplaces DNA",Airbnb,United States,Greenhouse,95,"Strong alignment with data engineering skills, including Python, SQL, and cloud technologies.","[Kafka, Hadoop]",Open Job
2,Data Engineer,Dropbox,Remote - Mexico,Greenhouse,90,Relevant experience in data engineering and cloud platforms like AWS and GCP.,"[Scala, Kafka]",Open Job
3,"Data Engineering Manager, Communication & Connectivity Data",Airbnb,Remote - US,Greenhouse,88,Leadership role that requires strong data engineering skills and experience with cloud technologies.,"[Team management, Advanced ML techniques]",Open Job
4,"Staff Data Engineer, Analytics Data Engineering",Dropbox,Remote - Canada: Select locations,Greenhouse,87,Focus on analytics and data engineering aligns well with the candidate's experience.,"[Kafka, Hadoop]",Open Job
5,"Staff Data Engineer, tvScientific",Pinterest,"San Francisco, CA, US; Remote, US",Greenhouse,85,"Strong emphasis on data engineering and cloud technologies, matching the candidate's skills.","[Scala, Kafka]",Open Job
6,Data Engineer,Figma,"San Francisco, CA • New York, NY • United States",Greenhouse,84,"Relevant experience in data engineering and cloud platforms, with a focus on analytics.","[Scala, Kafka]",Open Job
7,Principal Machine Learning & Data Engineer,Twilio,Remote - US,Greenhouse,82,"Combines data engineering with machine learning, leveraging the candidate's AI/ML skills.","[Team management, Advanced ML techniques]",Open Job
8,"Staff Data Engineer, Analytics Data Engineering",Dropbox,Remote - US: Select locations,Greenhouse,81,Focus on analytics and data engineering aligns well with the candidate's experience.,"[Kafka, Hadoop]",Open Job
9,"Staff Software Engineer, Data Engineering",Airbnb,United States,Greenhouse,80,"Strong focus on data engineering and software development, relevant to the candidate's skills.","[Kafka, Hadoop]",Open Job
10,Data Engineer,Dropbox,Remote - Mexico,Greenhouse,79,Relevant experience in data engineering and cloud platforms like AWS and GCP.,"[Scala, Kafka]",Open Job
